In [1]:
!pip install -q nltk scikit-learn matplotlib seaborn wordcloud plotly kaleido

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import re
import warnings

warnings.filterwarnings("ignore")
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.1 MB/s eta 0:00:00


In [2]:
nltk.download("vader_lexicon")
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

data_path = "kaggle_RC_2019-05.csv"
out_dir = Path("output")
chart_dir = out_dir / "charts"
table_dir = out_dir / "tables"
report_dir = out_dir / "report"

for p in [out_dir, chart_dir, table_dir, report_dir]:
    p.mkdir(parents=True, exist_ok=True)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip().str.lower()

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)


need = ["body", "subreddit"]
for col in need:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")


Columns: ['subreddit', 'body', 'controversiality', 'score']
Shape: (1000000, 4)


In [4]:
# CLEANING

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in word_tokenize(text) if w.isalpha() and w not in stop_words and len(w) > 2]
    return " ".join(tokens)

df["clean"] = df["body"].apply(clean_text)

In [5]:
# remove empty rows
df = df[df["clean"].str.len() > 0].copy()


In [6]:
# Sentiment
sia = SentimentIntensityAnalyzer()
scores = df["clean"].apply(sia.polarity_scores)
df["sentiment"] = scores.apply(lambda x: x["compound"])
df["sent_pos"] = scores.apply(lambda x: x["pos"])
df["sent_neg"] = scores.apply(lambda x: x["neg"])
df["sent_neu"] = scores.apply(lambda x: x["neu"])

def bucket_sentiment(x):
    if x >= 0.05:
        return "positive"
    elif x <= -0.05:
        return "negative"
    return "neutral"

df["sentiment_label"] = df["sentiment"].apply(bucket_sentiment)


In [7]:
# Core metrics
total = len(df)
sub_count = df["subreddit"].nunique()
avg_sent = df["sentiment"].mean()
med_sent = df["sentiment"].median()
pos_rate = (df["sentiment_label"] == "positive").mean()
neg_rate = (df["sentiment_label"] == "negative").mean()
neu_rate = (df["sentiment_label"] == "neutral").mean()

print("\nCore metrics")
print("Total comments:", total)
print("Subreddits:", sub_count)
print("Average sentiment:", round(avg_sent, 4))
print("Median sentiment:", round(med_sent, 4))
print("Positive rate:", round(pos_rate, 4))
print("Negative rate:", round(neg_rate, 4))
print("Neutral rate:", round(neu_rate, 4))



Core metrics
Total comments: 974427
Subreddits: 40
Average sentiment: 0.0877
Median sentiment: 0.0
Positive rate: 0.4503
Negative rate: 0.3001
Neutral rate: 0.2495


In [8]:
# top words

vec = CountVectorizer(max_features=50, ngram_range=(1, 2))
x = vec.fit_transform(df["clean"])
word_sum = x.sum(axis=0).A1
word_list = vec.get_feature_names_out()
top_words = pd.DataFrame({"term": word_list, "count": word_sum}).sort_values("count", ascending=False).head(20)

top_words.to_csv(table_dir / "top_words.csv", index=False)

In [9]:
# top positive / negative terms by sentiment

def get_terms(frame, n=20):
    v = CountVectorizer(max_features=100, ngram_range=(1, 2))
    x = v.fit_transform(frame["clean"])
    s = x.sum(axis=0).A1
    t = v.get_feature_names_out()
    return pd.DataFrame({"term": t, "count": s}).sort_values("count", ascending=False).head(n)

pos_terms = get_terms(df[df["sentiment_label"] == "positive"])
neg_terms = get_terms(df[df["sentiment_label"] == "negative"])

pos_terms.to_csv(table_dir / "positive_terms.csv", index=False)
neg_terms.to_csv(table_dir / "negative_terms.csv", index=False)

In [18]:
positive_terms = pd.read_csv("/content/output/tables/positive_terms.csv")
print(positive_terms)

       term   count
0      like  102492
1     would   47814
2       one   44610
3    people   44010
4      good   43101
5    please   41794
6       get   40895
7      time   37929
8     think   37043
9      make   36282
10     post   32724
11     know   30256
12  comment   29929
13   really   28523
14     even   27031
15     well   26788
16  message   26112
17     want   25725
18     need   25357
19     also   25235


In [21]:
negative_terms = pd.read_csv("/content/output/tables/negative_terms.csv")
print(negative_terms)

      term  count
0   people  41519
1     like  37286
2    would  35553
3      one  30410
4      get  29412
5    think  25014
6     shit  23692
7     even  23062
8     time  20236
9     know  19582
10    fuck  17247
11     bad  17182
12  really  16927
13     see  16336
14    also  16189
15   could  15941
16    make  15296
17     way  15273
18     got  15080
19   going  14901


In [10]:
# Subreddit summary

sub = df.groupby("subreddit").agg(
    comments=("body", "size"),
    avg_sentiment=("sentiment", "mean"),
    median_sentiment=("sentiment", "median"),
    avg_score=("score", "mean"),
    avg_controversiality=("controversiality", "mean")
).reset_index()

sub["pos_rate"] = df.groupby("subreddit")["sentiment_label"].apply(lambda s: (s == "positive").mean()).values
sub["neg_rate"] = df.groupby("subreddit")["sentiment_label"].apply(lambda s: (s == "negative").mean()).values
sub["risk"] = sub["avg_controversiality"].fillna(0) * (1 - sub["avg_sentiment"].fillna(0))
sub["opportunity"] = sub["avg_sentiment"].fillna(0) * np.log1p(sub["comments"])

sub = sub.sort_values("comments", ascending=False)
sub.to_csv(table_dir / "subreddit_summary.csv", index=False)

In [11]:
# Engagement analysis

eng = df[["sentiment", "score", "controversiality"]].copy()
eng["score"] = pd.to_numeric(eng["score"], errors="coerce")
eng["controversiality"] = pd.to_numeric(eng["controversiality"], errors="coerce")
eng = eng.dropna(subset=["score", "controversiality"])

corr = eng.corr(numeric_only=True)
corr.to_csv(table_dir / "correlation.csv")

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# score model

if len(eng) > 1000:
    X = eng[["sentiment", "controversiality"]].fillna(0)
    y = eng["score"].fillna(0)
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    model_mae = mean_absolute_error(y_test, pred)
    model_r2 = r2_score(y_test, pred)
    importance = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
else:
    model_mae = np.nan
    model_r2 = np.nan
    importance = pd.DataFrame(columns=["feature", "importance"])

importance.to_csv(table_dir / "feature_importance.csv", index=False)

In [23]:
# Topic modeling

topic_n = 6
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=10, max_df=0.8)
tx = tfidf.fit_transform(df["clean"])
lda = LatentDirichletAllocation(n_components=topic_n, random_state=42, learning_method="batch")
lda.fit(tx)

terms = tfidf.get_feature_names_out()
topic_rows = []
for i, comp in enumerate(lda.components_):
    idx = comp.argsort()[-12:][::-1]
    words = [terms[j] for j in idx]
    topic_rows.append({"topic": i + 1, "terms": ", ".join(words)})

topics = pd.DataFrame(topic_rows)
topics.to_csv(table_dir / "topics.csv", index=False)

doc_topic = lda.transform(tx)
df["topic"] = doc_topic.argmax(axis=1) + 1
topic_summary = df.groupby("topic").agg(
    comments=("body", "size"),
    avg_sentiment=("sentiment", "mean"),
    avg_score=("score", "mean")
).reset_index()
topic_summary.to_csv(table_dir / "topic_summary.csv", index=False)

In [27]:
# Visualizations
def save_fig(name):
    path = chart_dir / name
    plt.savefig(path, bbox_inches="tight")
    plt.close()

# sentiment distribution
plt.figure(figsize=(10, 5))
sns.histplot(df["sentiment"], bins=40, kde=True, color="steelblue")
plt.axvline(avg_sent, color="red", linestyle="--", label=f"mean={avg_sent:.3f}")
plt.title("Sentiment distribution")
plt.xlabel("VADER compound score")
plt.ylabel("Count")
plt.legend()
save_fig("sentiment_distribution.png")

# sentiment mix
mix = df["sentiment_label"].value_counts().reindex(["positive", "neutral", "negative"])
plt.figure(figsize=(8, 5))
sns.barplot(x=mix.index, y=mix.values, palette=["green", "gray", "red"])
plt.title("Sentiment mix")
plt.xlabel("Label")
plt.ylabel("Count")
save_fig("sentiment_mix.png")

# top subreddits by volume
top_sub = df["subreddit"].value_counts().head(15)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_sub.values, y=top_sub.index, palette="Blues_r")
plt.title("Top subreddits by volume")
plt.xlabel("Comments")
plt.ylabel("Subreddit")
save_fig("top_subreddits.png")

# subreddit sentiment
top_sent = sub.sort_values("comments", ascending=False).head(15).sort_values("avg_sentiment")
plt.figure(figsize=(12, 6))
sns.barplot(data=top_sent, x="avg_sentiment", y="subreddit", palette="coolwarm")
plt.title("Average sentiment by top subreddits")
plt.xlabel("Average sentiment")
plt.ylabel("Subreddit")
save_fig("subreddit_sentiment.png")


# word cloud
from wordcloud import WordCloud

text = " ".join(df["clean"].tolist())
wc = WordCloud(width=1400, height=700, background_color="white", max_words=200).generate(text)
plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Common words")
save_fig("wordcloud.png")

# correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0)
plt.title("Correlation heatmap")
save_fig("correlation_heatmap.png")

# topic chart
plt.figure(figsize=(12, 5))
sns.barplot(data=topic_summary.sort_values("comments", ascending=False), x="topic", y="comments", palette="viridis")
plt.title("Topic volume")
plt.xlabel("Topic")
plt.ylabel("Comments")
save_fig("topic_volume.png")

# risk vs opportunity
plot_df = sub.copy()
plot_df["size"] = np.sqrt(plot_df["comments"])
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=plot_df.head(30),
    x="risk",
    y="opportunity",
    size="comments",
    hue="avg_sentiment",
    palette="coolwarm",
    sizes=(50, 800),
    alpha=0.8
)
plt.title("Subreddit risk vs opportunity")
plt.xlabel("Risk")
plt.ylabel("Opportunity")
save_fig("risk_opportunity.png")

